# 06 -- Parameter stability

Paired script: `analysis/parameter_stability.py` (real, tested pipeline -- **fixed,
2026-07-22 Codex review finding:** this notebook previously computed its own sweep INLINE
with no CSV/JSON output and no pipeline test, and had a real statistical defect: it compared
`mean_r_diff_when_triggered` across different `giveback_percent` settings, but each setting
triggers on a DIFFERENT subset of paths -- averaging only each setting's own triggered
subset and comparing those means across settings is not a like-for-like comparison).
`parameter_stability.sweep_giveback_percent` now computes the mean over the SAME FULL set of
paths at every setting (a 0.0 contribution when the guard never triggers), so every row is
genuinely comparable to every other row.

**Fixed, 2026-07-22 Codex review finding (third round):** `run()` previously accepted only
caller-created in-memory R paths with no explicit-input CLI or dataset hash -- it now reads
R-paths from a real `r_paths.csv` (documented schema: `path_id, bar_index, r_value`), hashed
via the same `build_report_metadata` every other pipeline uses.

This notebook sweeps a REAL strategy parameter -- the V6.37-style giveback guard's
`giveback_percent` -- across a small grid and reports how the guard's behaviour (trigger
rate, mean R saved/lost over the full path set) changes.

**Uses clearly-labelled SYNTHETIC R-paths.** Real-data run: PENDING.

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.parameter_stability import run

In [2]:
# Hand-traced against ExitManager.mqh's EM_ShouldGivebackCloseV637 formula --
# the exact same fixture and hand-derivation as tests/test_parameter_stability.py.
#
# Path B = [0.0, 2.0, 1.0, 0.5], actual_final_r = 0.5 (last value):
#   pct=40: trigger_r = 2.0*0.6 = 1.2 -> triggers at index2 (current=1.0<=1.2), r=1.0
#           r_diff = 1.0 - 0.5 = 0.5
#   pct=60: trigger_r = 2.0*0.4 = 0.8 -> triggers at index3 (current=0.5<=0.8), r=0.5
#           r_diff = 0.5 - 0.5 = 0.0
#   pct=80: trigger_r = 2.0*0.2 = 0.4 -> never triggers (0.5 > 0.4 at index3); r_diff = 0.0
#
# Path C = [0.0, 1.25, 1.25], actual_final_r = 1.25:
#   peak reaches exactly the arm threshold but current==trigger_r never holds for
#   any percent > 0 -> never triggers at any swept percent. r_diff = 0.0 always.
PATH_B = [0.0, 2.0, 1.0, 0.5]
PATH_C = [0.0, 1.25, 1.25]

tmp_dir = Path(tempfile.mkdtemp(prefix="themba_paramstability_demo_"))
r_paths_csv = tmp_dir / "r_paths.csv"
rows = []
for path_id, values in {"pB": PATH_B, "pC": PATH_C}.items():
    for bar_index, r_value in enumerate(values):
        rows.append({"path_id": path_id, "bar_index": bar_index, "r_value": r_value})
pd.DataFrame(rows).to_csv(r_paths_csv, index=False)

stability_table = run(
    r_paths_csv,
    [40.0, 60.0, 80.0],
    output_csv=tmp_dir / "stability.csv",
    summary_json=tmp_dir / "summary.json",
    repo_path=PROJECT_ROOT.parents[1],
)
print(stability_table)

by_pct = stability_table.set_index("giveback_percent")
assert by_pct.loc[40.0, "n_triggered"] == 1
assert abs(by_pct.loc[40.0, "mean_r_diff_over_all_paths"] - 0.25) < 1e-9  # (0.5 + 0.0) / 2
assert by_pct.loc[60.0, "n_triggered"] == 1
assert abs(by_pct.loc[60.0, "mean_r_diff_over_all_paths"] - 0.0) < 1e-9
assert by_pct.loc[80.0, "n_triggered"] == 0
assert abs(by_pct.loc[80.0, "mean_r_diff_over_all_paths"] - 0.0) < 1e-9
# Every row's mean is over the SAME full 2-path set -- never a shrinking
# triggered-only subset (the Codex review finding this notebook now fixes).
assert (stability_table["n_paths"] == 2).all()

   giveback_percent  n_paths  n_triggered  mean_r_diff_over_all_paths  \
0              40.0        2            1                        0.25   
1              60.0        2            1                        0.00   
2              80.0        2            0                        0.00   

   r_diff_ci_lower  r_diff_ci_upper  
0              0.0              0.5  
1              0.0              0.0  
2              0.0              0.0  


## V8.11 and V6.37 2-D grid sweeps (Codex review findings, 2026-07-22, fifth/sixth round)

**Added, fifth round:** the cell above only ever swept V6.37's own `giveback_percent`
at a single fixed `arm_rr` -- `profit_giveback_diagnosis_plan.md` requires neighbouring
sweeps of BOTH controls for EACH model. `sweep_v811_arm_and_floor`/`run_v811_sweep` added
a genuine 2-D grid over V8.11's own two controls (`arm_r`, `floor_r`).

**Added, sixth round:** the module's own docstring had disclosed (and the canonical docs
then wrongly cited as "fully resolved") that the ANALOGOUS 2-D grid for V6.37's own two
controls (`arm_rr`, `giveback_percent` together) was still missing --
`sweep_v637_arm_rr_and_giveback_percent`/`run_v637_2d_sweep` close that gap. Both new
sweeps are exercised below, on the SAME `r_paths.csv` fixture as the cell above, not just
covered by `pytest` -- this notebook previously never ran either of them.

In [3]:
from analysis.parameter_stability import run_v637_2d_sweep, run_v811_sweep

# --- V8.11 (arm_r, floor_r) grid -- hand-traced against
# EM_ShouldGivebackCloseV811 (effective_arm=max(0.3,arm_r),
# effective_floor=max(0.0,floor_r), current<=effective_floor once armed):
# Path B=[0.0,2.0,1.0,0.5], arm_r=0.8, floor_r=1.5: peak=2.0>=0.8 -> armed
# at idx1; idx1 current=2.0<=1.5? No; idx2 current=1.0<=1.5? Yes -> triggers
# r=1.0, r_diff=1.0-0.5=0.5. Path C=[0.0,1.25,1.25]: peak=1.25>=0.8 -> armed
# at idx1; current=1.25<=1.5? Yes -> triggers r=1.25, r_diff=0.0.
v811_table = run_v811_sweep(
    r_paths_csv,
    [0.8],
    [1.5],
    output_csv=tmp_dir / "v811_stability.csv",
    summary_json=tmp_dir / "v811_summary.json",
    repo_path=PROJECT_ROOT.parents[1],
)
print(v811_table)
v811_row = v811_table.iloc[0]
assert v811_row["n_triggered"] == 2
assert abs(v811_row["mean_r_diff_over_all_paths"] - 0.25) < 1e-9  # (0.5 + 0.0) / 2

# --- V6.37 (arm_rr, giveback_percent) 2-D grid -- hand-traced against
# EM_ShouldGivebackCloseV637, over the SAME 2-path (pB, pC) set as above:
# arm_rr=1.25 (effective_arm=1.25): Path B peak=2.0>=1.25 -> arms; pct=40
# -> trigger_r=1.2, triggers idx2 (1.0<=1.2), r_diff=1.0-0.5=0.5. Path C
# peak=1.25>=1.25 -> arms at idx1, but current(1.25)<=trigger_r(0.75)?
# No, and no bar follows -> never triggers, r_diff=0.0. Mean over both =
# (0.5+0.0)/2=0.25, n_triggered=1. arm_rr=2.5 (effective_arm=2.5): NEITHER
# path's peak (2.0, 1.25) reaches 2.5 -> neither ever arms; r_diff=0.0 for
# both regardless of pct, n_triggered=0.
v637_2d_table = run_v637_2d_sweep(
    r_paths_csv,
    [1.25, 2.5],
    [40.0],
    output_csv=tmp_dir / "v637_2d_stability.csv",
    summary_json=tmp_dir / "v637_2d_summary.json",
    repo_path=PROJECT_ROOT.parents[1],
)
print(v637_2d_table)
by_arm_rr = v637_2d_table.set_index("arm_rr")
assert by_arm_rr.loc[1.25, "n_triggered"] == 1
assert abs(by_arm_rr.loc[1.25, "mean_r_diff_over_all_paths"] - 0.25) < 1e-9
assert by_arm_rr.loc[2.5, "n_triggered"] == 0
assert abs(by_arm_rr.loc[2.5, "mean_r_diff_over_all_paths"] - 0.0) < 1e-9

   arm_r  floor_r  n_paths  n_triggered  mean_r_diff_over_all_paths  \
0    0.8      1.5        2            2                        0.25   

   r_diff_ci_lower  r_diff_ci_upper  
0              0.0              0.5  


   arm_rr  giveback_percent  n_paths  n_triggered  mean_r_diff_over_all_paths  \
0    1.25              40.0        2            1                        0.25   
1    2.50              40.0        2            0                        0.00   

   r_diff_ci_lower  r_diff_ci_upper  
0              0.0              0.5  
1              0.0              0.0  


## Real-data run: PENDING

A meaningful parameter-stability read needs many real R-paths (real trade histories with
real bar-level MFE tracking), which do not exist yet -- see `analyse_giveback.py`'s own
"Real-data run: PENDING" note (notebook 02).